# 03 · EDA and Feature Engineering
**Reads:** `data/processed/flights_clean.parquet`, `data/raw/airlines.csv`  
**Writes:** `data/processed/train.parquet`, `data/processed/test.parquet`, `model/feature_config.joblib`

In [ ]:
import os, json, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
import joblib
from sklearn.model_selection import train_test_split, KFold

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


In [ ]:
DELAY_MINUTES = 15
df = pd.read_parquet(DATA_PROC / "flights_clean.parquet")
airlines = pd.read_csv(DATA_RAW / "airlines.csv")
print(df.shape)

## Feature engineering (time / route features)

In [ ]:
df["DEP_MIN"] = (df.SCHEDULED_DEPARTURE // 100 % 24) * 60 + df.SCHEDULED_DEPARTURE % 100   # minutes after midnight
df["ARR_MIN"] = (df.SCHEDULED_ARRIVAL // 100 % 24) * 60 + df.SCHEDULED_ARRIVAL % 100
df["DEP_HOUR"] = df.DEP_MIN // 60
df["ARR_HOUR"] = df.ARR_MIN // 60
df["IS_WEEKEND"] = df.DAY_OF_WEEK.isin([6, 7]).astype(int)
df["ROUTE"] = df.ORIGIN_AIRPORT + "_" + df.DESTINATION_AIRPORT
df["ORIGIN_HOUR"] = df.ORIGIN_AIRPORT + "_" + df.DEP_HOUR.astype(str)

base = df.DELAYED.mean()
airline_names = airlines.set_index("IATA_CODE").AIRLINE
df[["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "ARR_HOUR", "IS_WEEKEND", "ROUTE", "ORIGIN_HOUR", "DELAYED"]].head()

## Exploratory data analysis

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for a, col, t in zip(ax, ["MONTH", "DAY_OF_WEEK", "DEP_HOUR"], ["month", "day of week (1 = Mon)", "scheduled departure hour"]):
    r = df.groupby(col).DELAYED.mean()
    a.plot(r.index, r.values, marker="o"); a.axhline(base, color="red", ls="--", lw=1)
    a.set_title(f"Delay rate by {t}"); a.set_ylabel("share delayed")
plt.tight_layout(); plt.show()
m, h = df.groupby("MONTH").DELAYED.mean(), df.groupby("DEP_HOUR").DELAYED.mean()
print(f"Worst month: {m.idxmax()} ({m.max():.1%}), best month: {m.idxmin()} ({m.min():.1%})")
print(f"Worst departure hour: {h.idxmax()}:00 ({h.max():.1%}), best hour: {h.idxmin()}:00 ({h.min():.1%})")

In [ ]:
# Airline
a = df.groupby("AIRLINE").DELAYED.agg(["mean", "size"]).sort_values("mean")
a.index = a.index.map(airline_names)
plt.figure(figsize=(9, 5)); plt.barh(a.index, a["mean"], color="#4C72B0"); plt.axvline(base, color="red", ls="--")
plt.title("Delay rate by airline"); plt.xlabel("share delayed"); plt.show()

# Weekday x hour heat-map
plt.figure(figsize=(13, 3.5))
sns.heatmap(df.pivot_table(index="DAY_OF_WEEK", columns="DEP_HOUR", values="DELAYED"), cmap="YlOrRd")
plt.title("Delay rate: weekday x departure hour"); plt.show()

In [ ]:
# Distance and airports
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
band = pd.cut(df.DISTANCE, [0, 500, 1000, 1500, 2500, 10000], labels=["<500", "500-999", "1000-1499", "1500-2499", "2500+"])
df.groupby(band).DELAYED.mean().plot.bar(ax=ax[0], rot=0, color="#55A868"); ax[0].axhline(base, color="red", ls="--")
ax[0].set_title("Delay rate by distance (miles)")
o = df.groupby("ORIGIN_AIRPORT").DELAYED.agg(["mean", "size"])
o = o[o["size"] >= o["size"].quantile(0.5)].sort_values("mean")           # busier half of the airports
pd.concat([o.head(8), o.tail(8)])["mean"].plot.barh(ax=ax[1], color="#C44E52"); ax[1].set_title("Best / worst origin airports")
plt.tight_layout(); plt.show()

## Train / test split and leakage-safe target encoding
The test set is split off **before** any rate is computed. Training rows get out-of-fold rates (5-fold); test rows use rates fitted on the full training set.

In [ ]:
train, test = train_test_split(df, test_size=0.2, stratify=df.DELAYED, random_state=SEED)
train, test = train.reset_index(drop=True), test.reset_index(drop=True)
del df
print("train:", train.shape, "| test:", test.shape)

prior = train.DELAYED.mean()
RATE_KEYS = {"AIRLINE_RATE": "AIRLINE", "ORIGIN_RATE": "ORIGIN_AIRPORT", "DEST_RATE": "DESTINATION_AIRPORT",
             "ROUTE_RATE": "ROUTE", "ORIGIN_HOUR_RATE": "ORIGIN_HOUR"}

def rate_table(keys, y, m=30):
    g = y.groupby(keys).agg(["sum", "count"])
    return (g["sum"] + prior * m) / (g["count"] + m)

rate_tables = {}                                            # saved later for the web app
kf = KFold(5, shuffle=True, random_state=SEED)
for new_col, key_col in RATE_KEYS.items():
    rate_tables[new_col] = rate_table(train[key_col], train.DELAYED)
    oof = np.full(len(train), prior)
    for tr_idx, va_idx in kf.split(train):
        t = rate_table(train[key_col].iloc[tr_idx], train.DELAYED.iloc[tr_idx])
        oof[va_idx] = train[key_col].iloc[va_idx].map(t).fillna(prior).values
    train[new_col] = oof
    test[new_col] = test[key_col].map(rate_tables[new_col]).fillna(prior)

FEATURES = ["MONTH", "DAY_OF_WEEK", "DEP_HOUR", "ARR_HOUR", "SCHEDULED_TIME", "DISTANCE", "IS_WEEKEND",
            "AIRLINE_RATE", "ORIGIN_RATE", "DEST_RATE", "ROUTE_RATE", "ORIGIN_HOUR_RATE"]
TARGET = "DELAYED"
print(len(FEATURES), "model features:", FEATURES)
train[FEATURES].describe().T.round(3)

## Save train / test data and the feature configuration

In [ ]:
train.to_parquet(DATA_PROC / "train.parquet", index=False)
test.to_parquet(DATA_PROC / "test.parquet", index=False)

# rate tables + prior + feature list are reused by the final model bundle (notebook 06) and the web app
joblib.dump({"rate_tables": rate_tables, "prior": float(prior), "features": FEATURES, "target": TARGET, "rate_keys": RATE_KEYS},
            MODEL_DIR / "feature_config.joblib")
print("Saved train/test to", DATA_PROC, "and feature config to", MODEL_DIR / "feature_config.joblib")